# TomTom Traffic Telemetrics & Dynamic Cost Estimation

### Purpose
Connects a real-time spatial routing engine with the structured data warehouse layer (`data/fuel_prices_clean.json`). This module queries live commercial traffic infrastructure to dynamically calculate multi-route travel costs, adjusting a vehicle's volumetric fuel expenditure based on live congestion delay penalties instead of relying on static approximations.

### Key Mechanics
* **Dynamic IO Validation Testing:** Leverages `os.path.exists()` against the structured `data/` asset partition path to catch missing files, using an automated fallback state machine to maintain calculation runtime stability even if upstream scraper jobs crash.
* **Regex Token Sanitation Processing:** Executes a targeted `.replace("₱", "")` string strip routine, transforming localized currency symbols into clean computational floats to prevent numerical syntax crashes during cost-ranking operations.
* **Target Filtering Matrix Logic:** Features configurable validation tags (`TARGET_BRAND`, `TARGET_FUEL`) to isolate specific retail metrics (such as *Caltex Diesel*) without modifying foundational routing functions.
* **Geospatial Deduplication Filtering:** Dynamically group-filters incoming route matrices by rounding distances to the nearest 100 meters (`round(distance_km, 1)`). This automatically drops redundant, overlapping highway clones to keep both console outputs and the user interface clear of confusion.
* **Live Thermodynamic Load Multipliers:** Evaluates engine idling penalties by mapping the ratio of theoretical free-flow travel time against live-traffic travel time **(Free-Flow Time ÷ Live Time)**, directly scaling vehicle efficiency thresholds **(km/L)** to calculate fuel waste in gridlock.
* **Toggleable Feature Group Layers:** Segregates distinct routes into individual, toggleable canvas overlays (`folium.FeatureGroup`) mapped to an interactive layer control panel, allowing users to independently isolate, view, or hide overlapping route geometry.

Note: OSRM alternatives=3 returns *up to* 3 routes depending on geography. The [:3] slicer safely handles cases where only 1 or 2 routes physically exist.

In [9]:
import os
import json
import requests
import folium
from dotenv import load_dotenv
import os


# --- 1. CONFIGURATION & DATA INGESTION ---
fuel_data_path = os.path.join("data", "fuel_prices_clean.json")
current_fuel_price = 75.00  

if os.path.exists(fuel_data_path):
    try:
        with open(fuel_data_path, "r", encoding="utf-8") as f:
            clean_records = json.load(f)
            
        TARGET_BRAND = "Caltex"
        TARGET_FUEL = "Diesel"
        
        matched_price = None
        for record in clean_records:
            if record["brand"].lower() == TARGET_BRAND.lower() and record["fuel_display_name"].lower() == TARGET_FUEL.lower():
                matched_price = float(record["current_price"].replace("₱", ""))
                break
                
        if matched_price:
            current_fuel_price = matched_price
            print(f"[Data Pipeline] Dynamic fuel price loaded: ₱{current_fuel_price:.2f}/L ({TARGET_BRAND} {TARGET_FUEL})")
        else:
            all_prices = [float(r["current_price"].replace("₱", "")) for r in clean_records]
            current_fuel_price = sum(all_prices) / len(all_prices)
    except Exception as e:
        print(f"[Pipeline Warning] Error processing data layer. Reason: {e}")

vehicle_efficiency = 12.0  

# This tells Python to find and open the hidden .env file
load_dotenv()

# This fetches the secret key from the .env file. Leave the text EXACTLY like this!
TOMTOM_API_KEY = os.getenv("TOMTOM_API_KEY")


# --- 2. CORE COST CALCULATION ENGINE ---
def calculate_trip_cost(distance_km, vehicle_km_per_liter, traffic_multiplier, fuel_price_per_liter):
    actual_efficiency = vehicle_km_per_liter * traffic_multiplier 
    liters_needed = distance_km / actual_efficiency
    return liters_needed * fuel_price_per_liter

# --- 3. TELEMETRY REQUEST & API INTEGRATION ---
origin_lat, origin_lon = 14.6565, 121.0315  # SM North EDSA
# NEW TEST DESTINATION: Market! Market!, BGC (Forces 3 distinctly different highway paths)
dest_lat, dest_lon = 14.5494, 121.0568      

tomtom_url = (
    f"https://api.tomtom.com/routing/1/calculateRoute/{origin_lat},{origin_lon}:{dest_lat},{dest_lon}/json"
    f"?key={TOMTOM_API_KEY}&traffic=true&departAt=now&routeType=fastest&maxAlternatives=2&computeTravelTimeFor=all&sectionType=traffic"
)

print("\nQuerying TomTom Traffic Telemetrics API...")
response = requests.get(tomtom_url)
data = response.json()

# --- 4. DATA PROCESSING & GEOSPATIAL DEDUPLICATION ---
if "routes" in data:
    raw_routes = []
    
    for route in data["routes"]:
        summary = route["summary"]
        distance_km = summary["lengthInMeters"] / 1000
        ideal_time = summary.get("noTrafficTravelTimeInSeconds", summary["travelTimeInSeconds"])
        live_time = summary["travelTimeInSeconds"]
        traffic_delay = summary.get("trafficDelayInSeconds", 0)
        
        traffic_multiplier = ideal_time / live_time if live_time > 0 else 1.0
        cost = calculate_trip_cost(distance_km, vehicle_efficiency, traffic_multiplier, current_fuel_price)
        route_coords = [[point['latitude'], point['longitude']] for point in route['legs'][0]['points']]
        
        raw_routes.append({
            "cost": cost,
            "distance_km": distance_km,
            "ideal_time": ideal_time,
            "live_time": live_time,
            "traffic_delay": traffic_delay,
            "traffic_multiplier": traffic_multiplier,
            "coords": route_coords,
            "sections": route.get("sections", [])
        })
        
    # Sort raw options by cost ascending
    raw_routes.sort(key=lambda x: x["cost"])
    
    # DEDUPLICATION LOGIC: Drop overlapping pipeline clones sharing the same rounded distance
    processed_routes = []
    seen_rounded_distances = set()
    
    for r in raw_routes:
        rounded_dist = round(r["distance_km"], 1) # Group paths within 100 meters of each other
        if rounded_dist not in seen_rounded_distances:
            seen_rounded_distances.add(rounded_dist)
            processed_routes.append(r)
        else:
            print(f"[Pipeline Filter] Dropped redundant overlapping route option ({r['distance_km']:.2f} km)")

    total_routes = len(processed_routes)
    all_map_points = [pt for r in processed_routes for pt in r["coords"]]

    # Base Map Initialization
    route_map = folium.Map(location=[origin_lat, origin_lon], tiles="cartodbpositron")
    
    folium.Marker([origin_lat, origin_lon], tooltip="START: SM North EDSA", icon=folium.Icon(color="blue", icon="play")).add_to(route_map)
    folium.Marker([dest_lat, dest_lon], tooltip="END: Market! Market!, BGC", icon=folium.Icon(color="red", icon="stop")).add_to(route_map)

    # --- 5. RENDER CHANNELS THROUGH FEATURE GROUPS ---
    # Draw backward so the cheapest route (rank 0) stays completely visible on top
    for rank, route in reversed(list(enumerate(processed_routes))):
        
        # Enforce exact visual rules based on distinct route counts
        if total_routes >= 3:
            if rank == 0:
                recommendation_color = "#28a745"   # Green
                tier_label = "Recommended (Cheapest)"
            elif rank == 1:
                recommendation_color = "#fd7e14"   # Orange
                tier_label = "Slightly Recommended"
            else:
                recommendation_color = "#dc3545"   # Red
                tier_label = "Not Recommended (Expensive)"
        else:
            # 2 Routes Only -> Strict binary mapping
            if rank == 0:
                recommendation_color = "#28a745"   # Green
                tier_label = "Recommended (Cheapest)"
            else:
                recommendation_color = "#dc3545"   # Red
                tier_label = "Not Recommended (Expensive)"

        # Display full telemetrics breakdown in terminal console
        print(f"\n[Rendering Layer {rank}] --- {tier_label.upper()} ---")
        print(f"Distance: {route['distance_km']:.2f} km")
        print(f"Free-Flow Time: {route['ideal_time'] / 60:.1f} mins | Current Est. Time: {route['live_time'] / 60:.1f} mins")
        print(f"Traffic Delay Penalty: {route['traffic_delay'] / 60:.1f} mins (Multiplier: {route['traffic_multiplier']:.2f}x)")
        print(f"Dynamic Estimated Cost: ₱{route['cost']:.2f}")
        
        # Build independent toggle layer map channel
        route_layer = folium.FeatureGroup(name=f"Route {rank + 1}: {tier_label} (₱{route['cost']:.2f})")
        
        # Draw background recommendation ranking tier track
        folium.PolyLine(
            locations=route["coords"],
            color=recommendation_color, 
            weight=8 if rank == 0 else 5,  
            opacity=0.9 if rank == 0 else 0.6,
            tooltip=f"{tier_label} | Cost: ₱{route['cost']:.2f}"
        ).add_to(route_layer)
        
        # Overlay Waze-style localized traffic gridlock segment dashes
        for section in route["sections"]:
            if section["sectionType"] == "TRAFFIC":
                start_idx = section["startPointIndex"]
                end_idx = section["endPointIndex"]
                magnitude = section.get("magnitudeOfDelay", 0)
                
                if magnitude >= 3:  
                    folium.PolyLine(
                        locations=route["coords"][start_idx:end_idx + 1],
                        color="#4a0007",  
                        weight=4,
                        opacity=1.0,
                        dash_array="6, 6",
                        tooltip=f"Traffic Incident (Level {magnitude})"
                    ).add_to(route_layer)
                    
        route_layer.add_to(route_map)

    # Render interactive UI layer toggle control pad
    folium.LayerControl(collapsed=False).add_to(route_map)

    # Scale camera frame to focus on the whole trip matrix bounds
    route_map.fit_bounds(all_map_points)
    
    output_path = "tomtom_live_traffic_map.html"
    route_map.save(output_path)
    print(f"\nMap file saved successfully to: '{output_path}'")
else:
    print("\n[Pipeline Error] Routing pipeline execution failed.")


Querying TomTom Traffic Telemetrics API...

[Rendering Layer 2] --- NOT RECOMMENDED (EXPENSIVE) ---
Distance: 22.05 km
Free-Flow Time: 43.4 mins | Current Est. Time: 43.6 mins
Traffic Delay Penalty: 0.0 mins (Multiplier: 0.99x)
Dynamic Estimated Cost: ₱138.59

[Rendering Layer 1] --- SLIGHTLY RECOMMENDED ---
Distance: 18.20 km
Free-Flow Time: 37.1 mins | Current Est. Time: 35.5 mins
Traffic Delay Penalty: 0.0 mins (Multiplier: 1.05x)
Dynamic Estimated Cost: ₱108.71

[Rendering Layer 0] --- RECOMMENDED (CHEAPEST) ---
Distance: 17.20 km
Free-Flow Time: 34.2 mins | Current Est. Time: 32.7 mins
Traffic Delay Penalty: 0.0 mins (Multiplier: 1.05x)
Dynamic Estimated Cost: ₱102.82

Map file saved successfully to: 'tomtom_live_traffic_map.html'
